In [ ]:
# !pip install datasets sentence-transformers rank-bm25 anthropic pandas scikit-learn tqdm torch

In [2]:
from anthropic import HUMAN_PROMPT, AI_PROMPT
from dotenv import load_dotenv
load_dotenv("env.txt")

True

In [3]:
os.environ['ANTHROPIC_API_KEY'] = ""
client = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))


RAG + scikit-learn

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from sentence_transformers import SentenceTransformer, util
from rank_bm25 import BM25Okapi
import anthropic
from sklearn.metrics import precision_score, recall_score, f1_score
from tqdm import tqdm

Scikit-learn with SQuAD

In [5]:
class RAGModel:
    def __init__(self, embedding_model="multi-qa-mpnet-base-dot-v1", use_gpu=True, claude_version="claude-3-5-sonnet-20240620"):
        self.device = "cuda" if use_gpu and torch.cuda.is_available() else "cpu"
        self.model = SentenceTransformer(embedding_model, device=self.device)
        self.documents = []
        self.embeddings = None
        self.bm25 = None
        self.anthropic_client = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))
        self.claude_version = claude_version

    def load_documents(self, documents, batch_size=64):
        self.documents = documents
        tokenized_docs = [doc.split() for doc in documents]
        self.bm25 = BM25Okapi(tokenized_docs)

        embeddings_list = []
        for i in tqdm(range(0, len(documents), batch_size), desc="Encoding Documents"):
            batch = documents[i:i + batch_size]
            batch_embeddings = self.model.encode(batch, convert_to_tensor=True, device=self.device)
            embeddings_list.append(batch_embeddings)
        self.embeddings = torch.cat(embeddings_list, dim=0)

    def retrieve(self, query, top_k=10, retrieval_method="hybrid"):
        if retrieval_method == "embedding":
            query_embedding = self.model.encode(query, convert_to_tensor=True, device=self.device)
            scores = util.pytorch_cos_sim(query_embedding, self.embeddings)[0]
            top_results = torch.topk(scores, k=top_k)
            return [(self.documents[idx], scores[idx].item()) for idx in top_results.indices]

        elif retrieval_method == "bm25":
            scores = self.bm25.get_scores(query.split())
            top_indices = np.argsort(scores)[-top_k:][::-1]
            return [(self.documents[i], scores[i]) for i in top_indices]

        elif retrieval_method == "hybrid":
            bm25_results = self.retrieve(query, top_k=top_k, retrieval_method="bm25")
            embedding_results = self.retrieve(query, top_k=top_k, retrieval_method="embedding")
            doc_scores = {doc: score * 0.4 for doc, score in bm25_results}
            for doc, score in embedding_results:
                doc_scores[doc] = doc_scores.get(doc, 0) + score * 0.6
            sorted_results = sorted(doc_scores.items(), key=lambda x: x[1], reverse=True)
            return sorted_results[:top_k]

    def generate_answer(self, query, context):
        prompt = f"""
        You have been tasked with answering the following query:
        <query>
        {query}
        </query>
        Based on the following context:
        <context>
        {context}
        </context>
        Provide a concise and accurate response.
        """
        response = self.anthropic_client.messages.create(
            model=self.claude_version,
            max_tokens=1024,
            messages=[{"role": "user", "content": prompt}],
            temperature=0
        )
        return response.content[0].text

# Step 4: Define Evaluation Function
def evaluate_retrieval(model, queries, ground_truths_list, contexts, top_k=10, retrieval_method="hybrid"):
    y_true_classification = []
    y_pred_classification = []
    mrr_scores = []

    for i, query in enumerate(tqdm(queries, desc="Evaluating Queries")):
        ground_truths = ground_truths_list[i]
        retrieved_docs = model.retrieve(query, top_k=top_k, retrieval_method=retrieval_method)

        relevance = [1 if any(gt.strip().lower() in doc.strip().lower() for gt in ground_truths) else 0 for doc, _ in retrieved_docs]
        mrr = 1 / (relevance.index(1) + 1) if 1 in relevance else 0
        mrr_scores.append(mrr)

        is_true = any(gt.strip().lower() in " ".join(contexts).lower() for gt in ground_truths)
        is_pred = any(relevance)
        y_true_classification.append(1 if is_true else 0)
        y_pred_classification.append(1 if is_pred else 0)

    skl_precision = precision_score(y_true_classification, y_pred_classification, zero_division=0)
    skl_recall = recall_score(y_true_classification, y_pred_classification, zero_division=0)
    skl_f1 = f1_score(y_true_classification, y_pred_classification, zero_division=0)
    skl_mrr = np.mean(mrr_scores)

    return {
        "SKL_Precision": skl_precision,
        "SKL_Recall": skl_recall,
        "SKL_F1": skl_f1,
        "SKL_MRR": skl_mrr
    }

# Step 5: Load SQuAD Dataset
squad_dataset = load_dataset("squad", split="validation")
contexts = squad_dataset["context"]
questions = squad_dataset["question"]
answers = [[ans["text"][0]] for ans in squad_dataset["answers"]]  # Wrap in list for consistency

# Step 6: Instantiate RAGModel and Load Documents
rag_model = RAGModel()
rag_model.load_documents(contexts)

# Step 7: Evaluate Retrieval Metrics
metrics = evaluate_retrieval(rag_model, questions[:100], answers[:100], contexts, top_k=10, retrieval_method="hybrid")
print(f"Metrics for Claude 3.5 Sonnet on SQuAD:\n{metrics}")

# Step 8: Save Results to CSV
df = pd.DataFrame.from_dict({"Claude 3.5 Sonnet on SQuAD": metrics}, orient="index")
df.to_csv("squad_retrieval_metrics.csv", index=True)
print("Results saved to squad_retrieval_metrics.csv")


Evaluating Queries: 100%|██████████| 100/100 [00:12<00:00,  7.91it/s]


Metrics for Claude 3.5 Sonnet on SQuAD:
{'SKL_Precision': 1.0, 'SKL_Recall': 0.83, 'SKL_F1': 0.907103825136612, 'SKL_MRR': 0.6458333333333333}
Results saved to squad_retrieval_metrics.csv


Scikit-learn with FiQA

In [6]:
class RAGModel:
    def __init__(self, embedding_model="multi-qa-mpnet-base-dot-v1", use_gpu=True, claude_version="claude-3-5-sonnet-20240620"):
        self.device = "cuda" if use_gpu and torch.cuda.is_available() else "cpu"
        self.model = SentenceTransformer(embedding_model, device=self.device)
        self.documents = []
        self.embeddings = None
        self.bm25 = None
        self.anthropic_client = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))
        self.claude_version = claude_version

    def load_documents(self, documents, batch_size=64):
        self.documents = documents
        tokenized_docs = [doc.split() for doc in documents]
        self.bm25 = BM25Okapi(tokenized_docs)

        embeddings_list = []
        for i in tqdm(range(0, len(documents), batch_size), desc="Encoding Documents"):
            batch = documents[i:i + batch_size]
            batch_embeddings = self.model.encode(batch, convert_to_tensor=True, device=self.device)
            embeddings_list.append(batch_embeddings)
        self.embeddings = torch.cat(embeddings_list, dim=0)

    def retrieve(self, query, top_k=10, retrieval_method="hybrid"):
        if retrieval_method == "embedding":
            query_embedding = self.model.encode(query, convert_to_tensor=True, device=self.device)
            scores = util.pytorch_cos_sim(query_embedding, self.embeddings)[0]
            top_results = torch.topk(scores, k=top_k)
            return [(self.documents[idx], scores[idx].item()) for idx in top_results.indices]

        elif retrieval_method == "bm25":
            scores = self.bm25.get_scores(query.split())
            top_indices = np.argsort(scores)[-top_k:][::-1]
            return [(self.documents[i], scores[i]) for i in top_indices]

        elif retrieval_method == "hybrid":
            bm25_results = self.retrieve(query, top_k=top_k, retrieval_method="bm25")
            embedding_results = self.retrieve(query, top_k=top_k, retrieval_method="embedding")
            doc_scores = {doc: score * 0.4 for doc, score in bm25_results}
            for doc, score in embedding_results:
                doc_scores[doc] = doc_scores.get(doc, 0) + score * 0.6
            sorted_results = sorted(doc_scores.items(), key=lambda x: x[1], reverse=True)
            return sorted_results[:top_k]

    def generate_answer(self, query, context):
        prompt = f"""
        You have been tasked with answering the following query:
        <query>
        {query}
        </query>
        Based on the following context:
        <context>
        {context}
        </context>
        Provide a concise and accurate response.
        """
        response = self.anthropic_client.messages.create(
            model=self.claude_version,
            max_tokens=1024,
            messages=[{"role": "user", "content": prompt}],
            temperature=0
        )
        return response.content[0].text

def evaluate_retrieval(model, queries, ground_truths_list, contexts, top_k=10, retrieval_method="hybrid"):
    y_true_classification = []
    y_pred_classification = []

    mrr_scores = []

    for i, query in enumerate(tqdm(queries, desc="Evaluating Queries")):
        ground_truths = ground_truths_list[i]
        retrieved_docs = model.retrieve(query, top_k=top_k, retrieval_method=retrieval_method)

        relevance = [1 if any(gt.strip().lower() in doc.strip().lower() for gt in ground_truths) else 0 for doc, _ in retrieved_docs]

        mrr = 1 / (relevance.index(1) + 1) if 1 in relevance else 0
        mrr_scores.append(mrr)

        is_true = any(gt.strip().lower() in " ".join(contexts).lower() for gt in ground_truths)
        is_pred = any(relevance)
        y_true_classification.append(1 if is_true else 0)
        y_pred_classification.append(1 if is_pred else 0)

    skl_precision = precision_score(y_true_classification, y_pred_classification, zero_division=0)
    skl_recall = recall_score(y_true_classification, y_pred_classification, zero_division=0)
    skl_f1 = f1_score(y_true_classification, y_pred_classification, zero_division=0)

    skl_mrr = np.mean(mrr_scores)

    return {
        "SKL_Precision": skl_precision,
        "SKL_Recall": skl_recall,
        "SKL_F1": skl_f1,
        "SKL_MRR": skl_mrr
    }


fiqa_dataset = load_dataset("explodinggradients/fiqa", split="baseline")
questions = fiqa_dataset["question"]
ground_truths_list = fiqa_dataset["ground_truths"]
contexts = fiqa_dataset["contexts"]
flattened_contexts = [ctx for group in contexts for ctx in group]

rag_model = RAGModel()
rag_model.load_documents(flattened_contexts)

metrics = evaluate_retrieval(rag_model, questions[:100], ground_truths_list[:100], flattened_contexts, top_k=10, retrieval_method="hybrid")
print("Full Metrics:")
print(metrics)

df = pd.DataFrame.from_dict({"Claude 3.5 Sonnet on FiQA": metrics}, orient="index")
df.to_csv("scikit_fiqa_retrieval_metrics.csv", index=True)
print("Results saved to scikit_fiqa_retrieval_metrics.csv")



/home/czz7bf/.local/lib/python3.11/site-packages/datasets/load.py:1491: FutureWarning: The repository for explodinggradients/fiqa contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/explodinggradients/fiqa
You can avoid this message in future by passing the argument `trust_remote_code=True`.
Passing `trust_remote_code=True` will be mandatory to load this dataset from the next major release of `datasets`.
  warnings.warn(
Evaluating Queries: 100%|██████████| 30/30 [00:00<00:00, 37.68it/s]

Full Metrics:
{'SKL_Precision': 1.0, 'SKL_Recall': 0.9285714285714286, 'SKL_F1': 0.9629629629629629, 'SKL_MRR': 0.4066666666666666}
Results saved to scikit_fiqa_retrieval_metrics.csv
